# Connect to Tools Gateway (MCP path — Module 3a)

::: **Use this notebook if you followed Module 3a.** For the AgentCore path (Module 3b),
use **notebook 04b** instead. :::

## The Auth Challenge

FAST's agent authenticates to FAST's own gateway using FAST's Cognito pool. Module 3a's Tools
Gateway expects JWTs from Module 3a's Cognito pool — a different identity provider.

Solution: register an **OAuth2 Credential Provider** in AgentCore Identity's Token Vault that
points at Module 3a's Cognito. Then tell the agent's `gateway.py` to use that provider.

Steps:

1. Look up Module 3a's M2M client + Cognito discovery URL
2. Create the credential provider `workshop-tools-gateway-auth`
3. Overwrite `tools/gateway.py` to read the provider name from SSM
4. Publish `/FAST-stack/gateway_url` and `/FAST-stack/gateway_credential_provider`
5. Widen the runtime's IAM policy so it can read the new OAuth2 secret
6. Redeploy and verify


## Step 1: Look Up Module 3a Credentials

Pull the M2M client ID + secret from Module 3a's exports and construct the Cognito OIDC
discovery URL.


In [ ]:
import boto3, json, os

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
cfn = boto3.client("cloudformation", region_name=REGION)
sm = boto3.client("secretsmanager", region_name=REGION)

exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

M2M_CLIENT_ID = exports["workshop-CognitoM2MClientId"]
M2M_SECRET_ARN = exports["workshop-CognitoM2MClientSecretArn"]
COGNITO_POOL_ID = exports["workshop-CognitoUserPoolId"]

m2m_secret = json.loads(sm.get_secret_value(SecretId=M2M_SECRET_ARN)["SecretString"])
M2M_CLIENT_SECRET = m2m_secret["client_secret"]

DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_POOL_ID}/.well-known/openid-configuration"

print(f"M2M Client ID:  {M2M_CLIENT_ID}")
print(f"Discovery URL:  {DISCOVERY_URL}")

## Step 2: Create the OAuth2 Credential Provider

Idempotent — if `workshop-tools-gateway-auth` already exists, the API returns a
`ConflictException` which we treat as success.


In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

PROVIDER_NAME = "workshop-tools-gateway-auth"

provider_config = {
    "customOauth2ProviderConfig": {
        "oauthDiscovery": {"discoveryUrl": DISCOVERY_URL},
        "clientId": M2M_CLIENT_ID,
        "clientSecret": M2M_CLIENT_SECRET,
    }
}

try:
    resp = agentcore.create_oauth2_credential_provider(
        name=PROVIDER_NAME,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput=provider_config,
    )
    print(f"Created provider: {resp.get('credentialProviderArn', PROVIDER_NAME)}")
except agentcore.exceptions.ConflictException:
    print(f"Provider '{PROVIDER_NAME}' already exists — OK")
except Exception as e:
    name = e.__class__.__name__
    msg = str(e)
    if "already exists" in msg.lower() or name == "ValidationException" and "exists" in msg.lower():
        print(f"Provider '{PROVIDER_NAME}' already exists — OK")
    else:
        raise


## Step 3: Overwrite `tools/gateway.py`

Replace the baseline gateway client with one that reads the provider name from SSM, so you can
repoint the agent at a different gateway without rebuilding the container.


In [ ]:
GATEWAY_PY = r'''"""AgentCore Gateway MCP client with OAuth2 authentication."""

import logging
import os

from bedrock_agentcore.identity.auth import requires_access_token
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient
from utils.ssm import get_ssm_parameter

logger = logging.getLogger(__name__)

_stack_name = os.environ.get("STACK_NAME", "")
try:
    _provider_name = get_ssm_parameter(f"/{_stack_name}/gateway_credential_provider")
    logger.info("[GATEWAY] Using credential provider from SSM: %s", _provider_name)
except Exception:
    _provider_name = os.environ.get("GATEWAY_CREDENTIAL_PROVIDER_NAME", "")
    logger.info("[GATEWAY] Using credential provider from env: %s", _provider_name)


@requires_access_token(provider_name=_provider_name, auth_flow="M2M", scopes=[])
def _fetch_gateway_token(access_token: str) -> str:
    """Fetch OAuth2 token for Gateway authentication via AgentCore Identity Token Vault."""
    return access_token


def create_gateway_mcp_client() -> MCPClient:
    """Create MCP client for AgentCore Gateway with OAuth2 authentication."""
    stack_name = os.environ.get("STACK_NAME")
    if not stack_name:
        raise ValueError("STACK_NAME environment variable is required")
    if not stack_name.replace("-", "").replace("_", "").isalnum():
        raise ValueError("Invalid STACK_NAME format")
    gateway_url = get_ssm_parameter(f"/{stack_name}/gateway_url")
    logger.info("[GATEWAY] URL: %s", gateway_url)
    return MCPClient(
        lambda: streamablehttp_client(
            url=gateway_url,
            headers={"Authorization": f"Bearer {_fetch_gateway_token()}"},
        ),
        prefix="gw",
    )
'''

import pathlib, ast
FAST_DIR = pathlib.Path("/workshop/fast-agent")
gateway_py = FAST_DIR / "patterns" / "strands-travel-agent" / "tools" / "gateway.py"
gateway_py.write_text(GATEWAY_PY)
ast.parse(GATEWAY_PY)
print(f"Wrote {gateway_py} and parses cleanly")


## Step 4: Publish Gateway URL + Provider Name to SSM

Point the agent at the `tools-gateway` from Module 3a.


In [ ]:
gateways = agentcore.list_gateways().get("items", [])
gw = next((g for g in gateways if g["name"] == "tools-gateway"), None)
if gw is None:
    raise RuntimeError("Gateway 'tools-gateway' not found — finish Module 3a before continuing")

GATEWAY_ID = gw["gatewayId"]
GATEWAY_URL = f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{REGION}.amazonaws.com/mcp"

ssm = boto3.client("ssm", region_name=REGION)
ssm.put_parameter(Name="/FAST-stack/gateway_url", Value=GATEWAY_URL, Type="String", Overwrite=True)
ssm.put_parameter(Name="/FAST-stack/gateway_credential_provider", Value=PROVIDER_NAME, Type="String", Overwrite=True)

print(f"Gateway URL (Module 3a):  {GATEWAY_URL}")
print(f"Credential provider:      {PROVIDER_NAME}")


## Step 5: Widen the Agent IAM Policy

By default FAST scopes Secrets Manager access to its own OAuth2 secret. Widen it to all
AgentCore Identity OAuth2 secrets so the runtime can read the new provider's secret.


In [ ]:
import re
backend_stack = FAST_DIR / "infra-cdk" / "lib" / "backend-stack.ts"
text = backend_stack.read_text()
wide = "bedrock-agentcore-identity!default/oauth2/*"

# Check for the NARROW runtime-gateway-auth pattern specifically.
# FAST v0.4.1 already contains the wide pattern elsewhere in this file (on the OAuth2
# provider Lambda's role, unrelated to the runtime role), so checking "wide in text"
# would incorrectly short-circuit and skip widening the runtime role's inline policy.
narrow_re = re.compile(r"bedrock-agentcore-identity!default/oauth2/[^\"\x27`]*runtime-gateway-auth\*")
narrow_hits = narrow_re.findall(text)

if not narrow_hits:
    print(f"Already patched: {backend_stack} (no narrow runtime-gateway-auth pattern found)")
else:
    new_text, n = narrow_re.subn(wide, text)
    backend_stack.write_text(new_text)
    print(f"Patched {backend_stack} ({n} narrow pattern(s) replaced)")
    verify = backend_stack.read_text()
    if narrow_re.search(verify):
        raise RuntimeError(
            f"Widening verification failed: narrow runtime-gateway-auth pattern still in {backend_stack}."
        )
    print(f"Verified: {backend_stack} no longer contains the narrow runtime pattern")

## Step 6: Redeploy

Touch `travel_agent.py` so CDK rebuilds the container, then `cdk deploy`.


In [ ]:
import os, subprocess

travel_py = FAST_DIR / "patterns" / "strands-travel-agent" / "travel_agent.py"
with travel_py.open("a") as f:
    f.write("# Tools Gateway integration (MCP path)\n")
print(f"Touched {travel_py}")

os.chdir(FAST_DIR / "infra-cdk")

# FAST's CDK owns /FAST-stack/gateway_url and resets it to FAST's built-in gateway on
# every `cdk deploy`. Use try/finally to re-apply overrides even if deploy fails
# partway through — partial deployments still overwrite SSM.
import boto3
_ssm = boto3.client("ssm", region_name=REGION)
try:
    r = subprocess.run(
        ["npx", "cdk", "deploy", "--require-approval", "never"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"cdk deploy failed (exit {r.returncode}).\n"
            f"stdout tail: {r.stdout[-2500:]}\nstderr tail: {r.stderr[-2500:]}"
        )
    print(r.stdout[-600:])
    print("Redeploy complete")
finally:
    _ssm.put_parameter(Name="/FAST-stack/gateway_url", Value=GATEWAY_URL, Type="String", Overwrite=True)
    _ssm.put_parameter(Name="/FAST-stack/gateway_credential_provider", Value=PROVIDER_NAME, Type="String", Overwrite=True)
    print("Re-applied /FAST-stack/gateway_url and /FAST-stack/gateway_credential_provider")

## Verify

Open the Amplify URL (printed by notebook 02) and send:

> Plan a trip from SFO to Tokyo for 2026-09-15 to 2026-09-18, 2 guests

The agent should call `search_flights` and `search_hotels` through the Module 3a gateway and
return real results. In the runtime logs you should see tool invocations like
`gw_tg-workshop-flights-mcp___search_flights`.

If the agent says it cannot find the tools, check:

1. `aws logs tail /aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent-*-DEFAULT --since 5m`
2. The IAM widening patch — `grep 'oauth2/\*' infra-cdk/lib/backend-stack.ts`
3. `ssm get-parameters --names /FAST-stack/gateway_url /FAST-stack/gateway_credential_provider`
